# PolyMEF: Polyline Simplification for Multi-Exposure Fusion

Involves measuring exposure metrics, modeling them as a polyline, and using line simplification to select key images.

In [50]:
import os

import cv2
import numpy as np
from scipy.stats import entropy

import mon

mon.init()

[18:27:53] `mon-dev` loaded in: 0.0008 seconds.                                                      ]8;id=197305;file:///Volumes/ssd_01/10_workspace/11_code/mon/shared/mon/mon/__init__.py\__init__.py]8;;\:]8;id=548888;file:///Volumes/ssd_01/10_workspace/11_code/mon/shared/mon/mon/__init__.py#30\30]8;;\

Constants

In [51]:
current_dir  = mon.Path(os.getcwd())
root_dir     = current_dir.parents[0]
data_dir     = root_dir / "data"
run_dir      = root_dir / "run"

image_dir    = "100"
image_dir    = data_dir / "sample"   / image_dir
output_dir   = run_dir  / "polyline" / image_dir.stem

Load images

In [52]:
image_files = sorted(list(image_dir.rglob("*")))
images      = [cv2.imread(str(f)) for f in image_files]
images      = [cv2.cvtColor(img, cv2.COLOR_BGR2RGB) for img in images]

Metrics

In [53]:
def compute_entropy(image: np.ndarray) -> float:
    """Calculate image entropy based on grayscale histogram."""
    # Convert to grayscale if not already
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # Compute histogram (256 bins for 8-bit image)
    hist, _ = np.histogram(image.flatten(), bins=256, range=[0, 256], density=True)
    # Compute entropy, avoiding log(0)
    hist = hist[hist > 0]
    return entropy(hist, base=2)

In [54]:
def compute_histogram_spread(image, percentiles=(5, 95)):
    """Calculate histogram spread between specified percentiles."""
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # Compute percentile range to avoid outliers
    lower, upper = np.percentile(image.flatten(), percentiles)
    return upper - lower

In [55]:
def compute_mean_luminance(image):
    """Calculate mean luminance of the image."""
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    return np.mean(image)

In [56]:
def compute_contrast(image):
    """Calculate contrast as standard deviation of pixel intensities."""
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    return np.std(image)

In [57]:
def compute_gradient_magnitude(image):
    """Calculate average gradient magnitude using Sobel filters."""
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # Compute gradients in x and y directions
    grad_x = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)

    # Compute gradient magnitude
    grad_magnitude = np.sqrt(grad_x**2 + grad_y**2)
    return np.mean(grad_magnitude)

In [58]:
def compute_well_exposedness(image, sigma=0.2):
    """Calculate well-exposedness metric based on Gaussian weighting."""
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # Normalize image to [0, 1]
    image  = image.astype(float) / 255.0

    # Compute well-exposedness using Gaussian weighting centered at 0.5
    weight = np.exp(-((image - 0.5) ** 2) / (2 * sigma ** 2))
    return np.mean(weight)

In [59]:
def normalize_metric(value, min_val, max_val):
    """Normalize a metric value to [0, 1]."""
    return (value - min_val) / (max_val - min_val) if max_val > min_val else 0.5


def compute_combined_metric(image, weights=None):
    """Compute combined metric using product of normalized individual metrics.
    Weights: dict with keys for each metric (default equal: 1.0).
    """
    if weights is None:
        weights = {
            "entropy"    : 1.0,
            "hist_spread": 1.0,
            "mean_lum"   : 1.0,
            "contrast"   : 1.0,
            "grad_mag"   : 1.0,
            "well_exp"   : 1.0
        }

    # Compute individual metrics (from previous code)
    ent         = compute_entropy(image)
    hist_spread = compute_histogram_spread(image)
    mean_lum    = compute_mean_luminance(image)
    cont        = compute_contrast(image)
    grad_mag    = compute_gradient_magnitude(image)
    well_exp    = compute_well_exposedness(image)

    # Assume min/max ranges (adjust based on your dataset)
    # Example ranges; compute over all images for accuracy
    norm_ent  = normalize_metric(ent,         0,   8)  # Entropy often 0-8 bits
    norm_hist = normalize_metric(hist_spread, 0, 255)
    norm_mean = normalize_metric(mean_lum,    0, 255)
    norm_cont = normalize_metric(cont,        0, 128)  # Std dev up to ~128
    norm_grad = normalize_metric(grad_mag,    0, 255)
    norm_well = normalize_metric(well_exp,    0,   1)  # Already ~[0,1]

    # Combined via weighted product
    combined = (
        norm_ent  ** weights["entropy"] *
        norm_hist ** weights["hist_spread"] *
        norm_mean ** weights["mean_lum"] *
        norm_cont ** weights["contrast"] *
        norm_grad ** weights["grad_mag"] *
        norm_well ** weights["well_exp"]
    )
    return combined

Visualize

In [60]:
def plot_metric(metric_values, metric_name, output_dir):
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 5))
    plt.plot(metric_values, marker="o")
    plt.title(f"{metric_name}")
    plt.xlabel("Image Index")
    plt.ylabel(metric_name)
    plt.grid(True)
    plt.savefig(output_dir / f"{metric_name}.jpg")
    plt.close()

Measure and Plot

In [61]:
# Measure
metrics = {
    "entropy"    : [],
    "hist_spread": [],
    "mean_lum"   : [],
    "contrast"   : [],
    "grad_mag"   : [],
    "well_exp"   : [],
    "combined"   : []
}
for img in images:
    img_metrics = {
        "entropy"    : compute_entropy(img),
        "hist_spread": compute_histogram_spread(img),
        "mean_lum"   : compute_mean_luminance(img),
        "contrast"   : compute_contrast(img),
        "grad_mag"   : compute_gradient_magnitude(img),
        "well_exp"   : compute_well_exposedness(img),
        "combined"   : compute_combined_metric(img)
    }
    for key in metrics.keys():
        metrics[key].append(img_metrics[key])

# Plot
output_dir.mkdir(parents=True, exist_ok=True)
for key, values in metrics.items():
    plot_metric(values, key, output_dir)